# Utils - QB - ImputationWindowCalculator

Ce notebook illustre et vérifie le comportement de la classe
`ImputationWindowCalculator` (`tsforecast/frequency/imputation_window.py`),
qui calcule la fenêtre temporelle où **toutes** les colonnes d'un jeu de
données ont une valeur réelle (directement ou via une sous-période couverte
par une observation plus basse fréquence — ex. une observation trimestrielle
couvre ses 3 mois), puis étend éventuellement cette fenêtre selon un seuil de
couverture et un scope d'imputation. C'est le moteur central de
`HighFrequencyImputer` (§3 de `high_frequency_imputer_review.md`).

**Périmètre** : seules les méthodes publiques sont testées ici :
- `fit(data)`
- `get_imputation_window_mask(data=None)`
- `get_mask_at_frequency(frequency)`
- `get_columns_with_coverage(start, end, entity=None)`

ainsi que les attributs publics qu'elles exposent
(`imputation_window_start_`, `imputation_window_end_`,
`imputation_window_mask_`, `coverage_by_date_`, `index_freq_`,
`column_coverage_`). Les méthodes privées (`_fit_ts`, `_fit_panel`,
`_compute_window`, `_build_index_freq_grid`, `_build_coverage_matrix`,
`_extend_backward`, `_extend_forward`, `_convert_mask_to_frequency`,
`_columns_with_coverage`, `_get_entity_row_mask`) ne sont pas testées
directement : leur comportement n'est illustré ici qu'au travers de ses
effets observables sur les méthodes publiques.

**Contrat général** (cf. docstring de classe, déjà consolidée par la revue
HFI, §3.1/§3.2/§3.4/§3.5 traités) :
- Pour un panel, **tous** les attributs dict sont indexés par le tuple
  d'entité exact rendu par `get_unique_panel_entities` — `('France',)`,
  jamais `'France'`, même à un seul niveau d'entité. `get_columns_with_coverage`
  et `get_mask_at_frequency` normalisent une clé fournie par l'appelant
  (`normalize_entity_key`) avant un accès direct : une entité inconnue lève un
  `KeyError`, pas un repli silencieux.
- La grille haute fréquence interne (`imputation_window_mask_`,
  `coverage_by_date_`, ...) peut déborder de la dernière date réellement
  présente dans les données (ex. une observation trimestrielle étend la
  grille sur ses 3 mois, même si seul le premier existe comme ligne) : ne
  **jamais** l'intersecter directement avec un masque calculé sur ses propres
  données — toujours passer par `get_imputation_window_mask(data)`, qui
  réaligne proprement.
- Ce module est déjà largement couvert par `tests/frequency/test_imputation_window.py`
  (90% de couverture, cf. mémoire de la revue HFI) : ce notebook sert de
  complément visuel/exploratoire à cette suite, pas de première découverte de
  bugs — un seul point de vigilance mineur, déjà connu et documenté comme
  hors périmètre, est signalé en §4.4 et repris dans la synthèse (§8, point 3).

## 1 - Import et instanciation

In [1]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe testée
from tsforecast.frequency.imputation_window import ImputationWindowCalculator

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation
calc = ImputationWindowCalculator()

print("ImputationWindowCalculator instancié avec succès !")


# Depuis [SPEC] 7.2 : sur panel, les trois masques et coverage_by_date_ sont
# une unique pd.Series a MultiIndex (entity..., date), plus un Dict[entity, Series].
# Ce helper en extrait la tranche d'une entite sur un index temporel simple.
def entity_slice(series, entity):
    """Tranche d'une entite depuis une Series a MultiIndex (entity..., date)."""
    rows = np.ones(len(series), dtype=bool)
    for level, value in enumerate(entity):
        rows &= (series.index.get_level_values(level) == value)
    if not rows.any():
        return None
    sub = series[rows]
    sub.index = sub.index.get_level_values(-1)
    return sub


ImputationWindowCalculator instancié avec succès !


## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (convention
déjà suivie dans `target_frequency_validator.ipynb`, `frequency_aligner.ipynb`,
`regularizer.ipynb` : aucune fonction partagée n'existe entre notebooks dans
ce projet) pour obtenir :
- `df_timeseries` : indicateurs macroéconomiques mensuels/trimestriels/annuels,
  avec une variable annuelle (`balance_commerciale_annuelle`) dont l'historique
  démarre avant la grille mensuelle.
- `df_panel` : mêmes indicateurs pour 3 pays (France, Allemagne, Italie), avec
  des périodes couvertes et des fréquences de publication hétérogènes par
  entité (`depenses_publiques_pib` est annuelle pour la France/l'Italie,
  trimestrielle pour l'Allemagne).

In [2]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle : historique antérieur à la grille mensuelle
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Historique limité de la production industrielle
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

print(f"df_timeseries : {df_timeseries.shape}, période {df_timeseries.index.min().date()} à {df_timeseries.index.max().date()}")
print(f"Colonnes : {list(df_timeseries.columns)}")
df_timeseries.tail(8)

df_timeseries : (82, 5), période 2015-01-01 à 2024-07-01
Colonnes : ['production_industrielle', 'inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle']


,production_industrielle,inflation_ipc,taux_chomage,pib_trimestriel,balance_commerciale_annuelle
date,,,,,
2023-12-01,114.460901,2.731558,8.174381,NaN,NaN
2024-01-01,113.792415,2.780858,7.815465,2805.907023,NaN
2024-02-01,117.885427,2.493428,7.927907,NaN,NaN
2024-03-01,112.899228,2.787625,7.816807,NaN,NaN
2024-04-01,118.655931,2.826383,7.600449,2220.907444,NaN
2024-05-01,117.344031,2.544669,8.082635,NaN,NaN
2024-06-01,115.859181,3.339220,7.703707,NaN,NaN
2024-07-01,115.137641,NaN,NaN,NaN,NaN


In [3]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2015-01-01'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle', 'annual_start_date': '2016-01-01'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2016-01-01'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle : historique antérieur à la grille mensuelle du pays
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

print(f"df_panel : {df_panel.shape}")
print(f"Colonnes : {list(df_panel.columns)}")
df_panel.loc['Allemagne'].tail(6)

df_panel : (225, 6)
Colonnes : ['production_industrielle', 'inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'depenses_publiques_pib', 'balance_commerciale_annuelle']


,production_industrielle,inflation_ipc,taux_chomage,pib_trimestriel,depenses_publiques_pib,balance_commerciale_annuelle
date,,,,,,
2023-11-01,114.009419,2.255189,6.308948,NaN,NaN,NaN
2023-12-01,112.914674,2.461804,6.374127,NaN,NaN,NaN
2024-01-01,111.444676,2.635840,6.008062,4139.230237,46.975029,NaN
2024-02-01,114.276673,2.106560,6.318388,NaN,NaN,NaN
2024-03-01,108.966618,2.098094,6.103817,NaN,NaN,NaN
2024-04-01,111.272356,NaN,NaN,NaN,NaN,NaN


### 2.2 - Jeux de données ciblés supplémentaires

`df_timeseries`/`df_panel` couvrent bien les cas réalistes (fréquences mixtes,
délais de publication, couverture hétérogène par entité), mais leur
complexité rend difficile l'isolation précise de certains mécanismes internes
(extension contiguë, seuil de couverture, cas où aucune fenêtre n'existe,
entité sans fréquence détectable...). Quelques jeux minimaux et contrôlés
sont donc construits en complément, sur le même principe que
`regularizer.ipynb` §2.2.

In [4]:
# --- TS : fenêtre stricte au centre, avec une "épaule" de couverture à 50% avant un
# trou total, pour isoler l'extension contiguë (mêmes proportions que le test
# unitaire `test_extension_stops_at_first_gap`) ---
dates_gap = pd.date_range('2020-01-01', periods=20, freq='MS')
# 'b' listée en premier : c'est la colonne CONTIGUË sur la fenêtre stricte, utilisée
# en interne pour détecter la position (S/E) de la grille -- une colonne à trous en
# première position ferait échouer cette détection (bug latent hors périmètre, §4.4).
df_gap = pd.DataFrame({'b': np.nan, 'a': np.nan}, index=dates_gap, dtype=float)
df_gap.loc[dates_gap[10:15], 'a'] = 1.0
df_gap.loc[dates_gap[10:15], 'b'] = 1.0  # fenêtre stricte : couverture totale sur [10, 15)
df_gap.loc[dates_gap[0:4], 'a'] = 1.0    # épaule avant le trou : couverture 50% sur [0, 4)
df_gap.loc[dates_gap[8:10], 'a'] = 1.0   # épaule juste avant la fenêtre stricte : 50% sur [8, 10)
# [4, 8) reste à couverture nulle : le "trou" qui doit bloquer toute extension au-delà

# --- TS : deux colonnes à couverture disjointe -> aucune période où les deux
# sont simultanément disponibles ---
dates_disjoint = pd.date_range('2020-01-01', periods=12, freq='MS')
df_disjoint = pd.DataFrame({'a': np.nan, 'b': np.nan}, index=dates_disjoint, dtype=float)
df_disjoint.loc[dates_disjoint[0:6], 'a'] = 1.0
df_disjoint.loc[dates_disjoint[6:12], 'b'] = 1.0

# --- TS : fenêtre stricte non contiguë (deux blocs de couverture totale séparés
# par un trou) ---
dates_nc = pd.date_range('2020-01-01', periods=12, freq='MS')
df_noncontiguous = pd.DataFrame({'a': 1.0, 'b': np.nan}, index=dates_nc, dtype=float)
df_noncontiguous.loc[dates_nc[[2, 3, 8, 9]], 'b'] = 1.0

# --- TS : fenêtre stricte réduite à une seule date ('a' couvre tout sauf le mois 5,
# 'b' n'a que 2 observations adjacentes aux mois 4 et 5 pour rester détectable) ---
dates_single = pd.date_range('2020-01-01', periods=12, freq='MS')
df_single_strict = pd.DataFrame({'a': range(12), 'b': np.nan}, index=dates_single, dtype=float)
df_single_strict.loc[dates_single[5], 'a'] = np.nan
df_single_strict.loc[dates_single[[4, 5]], 'b'] = [1.0, 2.0]

# --- Panel : une entité OK (couverture totale), une entité BAD à couverture
# disjointe (aucune fenêtre stricte pour elle) ---
dates_panel_edge = pd.date_range('2020-01-01', periods=12, freq='MS')
idx_ok = pd.MultiIndex.from_product([['OK'], dates_panel_edge], names=['entity', 'date'])
idx_bad = pd.MultiIndex.from_product([['BAD'], dates_panel_edge], names=['entity', 'date'])
df_ok = pd.DataFrame({'a': 1.0, 'b': 1.0}, index=idx_ok)
df_bad = pd.DataFrame({'a': np.nan, 'b': np.nan}, index=idx_bad, dtype=float)
df_bad.loc[('BAD', dates_panel_edge[0:6]), 'a'] = 1.0
df_bad.loc[('BAD', dates_panel_edge[6:12]), 'b'] = 1.0
panel_ok_bad = pd.concat([df_ok, df_bad])

# --- Panel : une entité à dates complètement irrégulières (fréquence indétectable,
# mais >= 2 observations donc pas d'erreur de détection) ---
irregular_dates = pd.to_datetime(['2020-01-01', '2020-01-05', '2020-03-11', '2020-08-02', '2021-01-30'])
idx_irregular = pd.MultiIndex.from_arrays(
    [['IRREGULAR'] * len(irregular_dates), irregular_dates], names=['entity', 'date']
)
df_irregular = pd.DataFrame(
    {'a': range(len(irregular_dates)), 'b': range(len(irregular_dates))}, index=idx_irregular, dtype=float
)
panel_irregular_entity = pd.concat([df_ok, df_irregular])

# --- Panel : une entité entièrement NaN sur toutes les colonnes (fréquence de
# l'index détectable, mais aucune donnée exploitable) ---
idx_empty = pd.MultiIndex.from_product([['OK', 'EMPTY'], dates_panel_edge], names=['entity', 'date'])
panel_empty_entity = pd.DataFrame({'a': 1.0, 'b': 1.0}, index=idx_empty)
panel_empty_entity.loc['EMPTY'] = np.nan

print("df_gap               :", df_gap.shape)
print("df_disjoint           :", df_disjoint.notna().sum().to_dict())
print("df_noncontiguous       :", df_noncontiguous['b'].dropna().index.tolist())
print("df_single_strict       : trou unique au mois 5, 'b' aux mois 4-5 uniquement")
print("panel_ok_bad            :", panel_ok_bad.shape)
print("panel_irregular_entity  :", set(panel_irregular_entity.index.get_level_values('entity')))
print("panel_empty_entity      :", set(panel_empty_entity.index.get_level_values('entity')))

df_gap               : (20, 2)
df_disjoint           : {'a': 6, 'b': 6}
df_noncontiguous       : [Timestamp('2020-03-01 00:00:00'), Timestamp('2020-04-01 00:00:00'), Timestamp('2020-09-01 00:00:00'), Timestamp('2020-10-01 00:00:00')]
df_single_strict       : trou unique au mois 5, 'b' aux mois 4-5 uniquement
panel_ok_bad            : (24, 2)
panel_irregular_entity  : {'OK', 'IRREGULAR'}
panel_empty_entity      : {'EMPTY', 'OK'}


## 3 - Construction : `ImputationWindowCalculator(coverage_threshold, imputation_scope, min_columns)`

Trois paramètres, tous validés dès `__init__` (avant tout `fit`) :
- `coverage_threshold` (défaut `0.5`) : doit être dans `[0, 1]`.
- `imputation_scope` (défaut `'strict'`) : un des 4 littéraux
  `'strict'`/`'extended_backward'`/`'extended_forward'`/`'extended_both'`.
- `min_columns` (défaut `2`) : au moins `2` (une fenêtre d'imputation n'a pas
  de sens avec une seule colonne).

In [5]:
# coverage_threshold hors de [0, 1]
try:
    ImputationWindowCalculator(coverage_threshold=1.5)
except ValueError as e:
    print("ValueError (coverage_threshold) :", e)

# imputation_scope invalide
try:
    ImputationWindowCalculator(imputation_scope='bogus')
except ValueError as e:
    print("\nValueError (imputation_scope) :", e)

# min_columns < 2
try:
    ImputationWindowCalculator(min_columns=1)
except ValueError as e:
    print("\nValueError (min_columns) :", e)

# Construction valide : uniquement les paramètres sont stockés, aucun attribut
# de fenêtre n'existe avant fit()
calc_valid = ImputationWindowCalculator(coverage_threshold=0.7, imputation_scope='extended_both', min_columns=3)
print("\nParamètres stockés :", calc_valid.coverage_threshold, calc_valid.imputation_scope, calc_valid.min_columns)
print("Attributs de fenêtre avant fit() :", calc_valid.imputation_window_start_, calc_valid.imputation_window_mask_)

ValueError (coverage_threshold) : coverage_threshold must be between 0 and 1, got 1.5

ValueError (imputation_scope) : imputation_scope must be one of 'strict', 'extended_backward', 'extended_forward', 'extended_both', got 'bogus'

Paramètres stockés : 0.7 extended_both 3
Attributs de fenêtre avant fit() : None None


## 4 - `fit(data)`

### 4.1 - Validation des données d'entrée

Rejetée avant toute détection de fréquence : type non-DataFrame, DataFrame
vide, index ni `DatetimeIndex` ni `MultiIndex`, nombre de colonnes `<
min_columns`.

In [6]:
calc_guard = ImputationWindowCalculator(min_columns=2)

# Type non-DataFrame
try:
    calc_guard.fit([1, 2, 3])
except ValueError as e:
    print("ValueError (non-DataFrame) :", e)

# DataFrame vide
try:
    calc_guard.fit(pd.DataFrame(index=pd.DatetimeIndex([])))
except ValueError as e:
    print("\nValueError (vide) :", e)

# Index sans notion temporelle (RangeIndex)
try:
    calc_guard.fit(pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]}))
except ValueError as e:
    print("\nValueError (index invalide) :", e)

# Moins de colonnes que min_columns
try:
    calc_guard.fit(pd.DataFrame({'a': range(12)}, index=pd.date_range('2023-01-01', periods=12, freq='MS')))
except ValueError as e:
    print("\nValueError (min_columns) :", e)

ValueError (non-DataFrame) : data must be a pandas DataFrame, got list

ValueError (vide) : data cannot be empty

ValueError (index invalide) : data must have a DatetimeIndex or MultiIndex with datetime level

ValueError (min_columns) : Data has 1 columns, but min_columns=2


### 4.2 - Séries temporelles : `fit` nominal sur `df_timeseries`

`fit` détecte la structure (TS ici, car l'index est un simple `DatetimeIndex`),
détecte la fréquence de chaque colonne et de l'index, puis calcule la fenêtre
stricte (coverage == 1.0 sur la grille haute fréquence). Tous les attributs
de fenêtre sont alors des **scalaires**/objets simples (pas de dict), à
l'inverse du cas panel (§4.3).

In [7]:
calc_ts = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='strict')
calc_ts.fit(df_timeseries)

print("imputation_window_start_ :", calc_ts.imputation_window_start_)
print("imputation_window_end_   :", calc_ts.imputation_window_end_)
print("index_freq_               :", calc_ts.index_freq_, "(type :", type(calc_ts.index_freq_).__name__, ")")
print("Type de imputation_window_mask_ :", type(calc_ts.imputation_window_mask_).__name__)
print("Nb dates actives / grille  :", calc_ts.imputation_window_mask_.sum(), "/", len(calc_ts.imputation_window_mask_))
print()
print("column_coverage_ (bornes [début, fin] par colonne, sur la grille) :")
for col, (start, end) in calc_ts.column_coverage_.items():
    print(f"  {col:32} -> [{start.date()}, {end.date()}]")
print()
print("La fenêtre stricte = intersection de toutes ces bornes :")
print("  max(débuts) =", max(s for s, _ in calc_ts.column_coverage_.values()).date(), "== imputation_window_start_")
print("  min(fins)   =", min(e for _, e in calc_ts.column_coverage_.values()).date(), "== imputation_window_end_")

imputation_window_start_ :

 2019-01-01 00:00:00
imputation_window_end_   : 2023-12-01 00:00:00
index_freq_               : M (type : str )
Type de imputation_window_mask_ : Series
Nb dates actives / grille  : 60 / 116

column_coverage_ (bornes [début, fin] par colonne, sur la grille) :
  production_industrielle          -> [2019-01-01, 2024-07-01]
  inflation_ipc                    -> [2018-01-01, 2024-06-01]
  taux_chomage                     -> [2018-01-01, 2024-06-01]
  pib_trimestriel                  -> [2018-01-01, 2024-06-01]
  balance_commerciale_annuelle     -> [2015-01-01, 2023-12-01]

La fenêtre stricte = intersection de toutes ces bornes :
  max(débuts) = 2019-01-01 == imputation_window_start_
  min(fins)   = 2023-12-01 == imputation_window_end_


### 4.3 - Panel : `fit` nominal sur `df_panel`

Pour un panel, **chaque** attribut de fenêtre devient un `dict` indexé par le
tuple d'entité exact (`get_unique_panel_entities`), y compris à un seul
niveau d'entité (`('France',)`, jamais `'France'`) — la fenêtre est calculée
**indépendamment par entité**, avec sa propre fréquence détectée.

In [8]:
calc_panel = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='strict')
calc_panel.fit(df_panel)

print("Toutes les clés sont des tuples :", all(isinstance(k, tuple) for k in calc_panel.imputation_window_start_))
print()
for entity in calc_panel.imputation_window_start_:
    print(
        f"{entity!r:16} -> fenêtre stricte [{calc_panel.imputation_window_start_[entity].date()}, "
        f"{calc_panel.imputation_window_end_[entity].date()}], index_freq_={calc_panel.index_freq_[entity]}"
    )

print()
print("column_coverage_ contient une entrée PAR ENTITÉ (pas seulement la dernière) :")
print("  Clés :", set(calc_panel.column_coverage_))
print("  Colonnes couvertes pour l'Allemagne :", list(calc_panel.column_coverage_[('Allemagne',)]))

Toutes les clés sont des tuples : True

('Allemagne',)   -> fenêtre stricte [2019-01-01, 2023-12-01], index_freq_=M
('France',)      -> fenêtre stricte [2018-06-01, 2023-12-01], index_freq_=M
('Italie',)      -> fenêtre stricte [2019-06-01, 2023-12-01], index_freq_=M

column_coverage_ contient une entrée PAR ENTITÉ (pas seulement la dernière) :
  Clés : {('Allemagne',), ('Italie',), ('France',)}
  Colonnes couvertes pour l'Allemagne : ['production_industrielle', 'inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'depenses_publiques_pib', 'balance_commerciale_annuelle']


### 4.4 - Effet de `imputation_scope` : extension contiguë du masque

Sur `df_gap` (fenêtre stricte sur `[10, 15)`, "épaules" à couverture 50% en
`[0, 4)` et `[8, 10)`, séparées par un trou total en `[4, 8)`) : l'extension
`'extended_backward'` doit s'arrêter **au premier trou**, sans jamais activer
de dates au-delà, même si elles satisferaient individuellement le seuil.

In [9]:
for scope in ['strict', 'extended_backward', 'extended_forward', 'extended_both']:
    calc_scope = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope=scope, min_columns=2)
    calc_scope.fit(df_gap)
    active = calc_scope.imputation_window_mask_.index[calc_scope.imputation_window_mask_]
    print(f"{scope:20} -> {len(active)} dates actives, bornes = [{active.min().date()}, {active.max().date()}]")

print()
# 'extended_backward' : l'épaule [8, 10) (juste avant la fenêtre stricte) est activée,
# mais PAS l'épaule [0, 4) (au-delà du trou [4, 8) à couverture nulle)
calc_backward = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='extended_backward')
calc_backward.fit(df_gap)
print("[0, 4)  activées ?", calc_backward.imputation_window_mask_.loc[dates_gap[0:4]].any(), "(doit être False)")
print("[8, 10) activées ?", calc_backward.imputation_window_mask_.loc[dates_gap[8:10]].all(), "(doit être True)")

strict               -> 5 dates actives, bornes = [2020-11-01, 2021-03-01]
extended_backward    -> 7 dates actives, bornes = [2020-09-01, 2021-03-01]
extended_forward     -> 5 dates actives, bornes = [2020-11-01, 2021-03-01]
extended_both        -> 7 dates actives, bornes = [2020-09-01, 2021-03-01]

[0, 4)  activées ? False (doit être False)
[8, 10) activées ? True (doit être True)


### 4.5 - Effet de `coverage_threshold`

Toujours sur `df_gap` : l'épaule `[8, 10)` a une couverture de 50%. Avec un
seuil `<= 0.5`, elle est activée ; avec un seuil `> 0.5`, l'extension
s'arrête net à la fenêtre stricte (aucune activation).

In [10]:
for threshold in [0.3, 0.5, 0.8]:
    calc_thr = ImputationWindowCalculator(coverage_threshold=threshold, imputation_scope='extended_backward')
    calc_thr.fit(df_gap)
    active = calc_thr.imputation_window_mask_.index[calc_thr.imputation_window_mask_]
    print(f"coverage_threshold={threshold} -> {len(active)} dates actives, début = {active.min().date()}")

coverage_threshold=0.3 -> 7 dates actives, début = 2020-09-01


coverage_threshold=0.5 -> 7 dates actives, début = 2020-09-01


coverage_threshold=0.8 -> 5 dates actives, début = 2020-11-01


### 4.6 - Avertissement : aucune fenêtre stricte (`df_disjoint`)

Quand deux colonnes n'ont **jamais** de couverture simultanée, `fit` émet un
`UserWarning`, `imputation_window_start_`/`_end_` valent `None`, mais
`imputation_window_mask_`/`coverage_by_date_`/`column_coverage_` restent
renseignés (masque entièrement `False`).

In [11]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    calc_none = ImputationWindowCalculator(coverage_threshold=0.5)
    calc_none.fit(df_disjoint)
    print("Warning :", w[0].category.__name__, "-", w[0].message)

print()
print("start/end :", calc_none.imputation_window_start_, calc_none.imputation_window_end_)
print("mask_ non None, entièrement False :", calc_none.imputation_window_mask_ is not None, "/",
      calc_none.imputation_window_mask_.sum(), "dates actives sur", len(calc_none.imputation_window_mask_))
print("coverage_by_date_ toujours calculée :", calc_none.coverage_by_date_.max(), "(max, jamais 1.0 ici)")


start/end : None None
mask_ non None, entièrement False : True / 0 dates actives sur 13
coverage_by_date_ toujours calculée : 0.5 (max, jamais 1.0 ici)


### 4.7 - Avertissement : fenêtre stricte non contiguë (`df_noncontiguous`)

Deux blocs de couverture totale séparés par un trou : `imputation_window_start_`/
`_end_` reportent les bornes `[min, max]` malgré la discontinuité (pas les
deux blocs séparément), avec un `UserWarning` explicite.

In [12]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    calc_nc = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='strict')
    calc_nc.fit(df_noncontiguous)
    print("Warning :", w[0].category.__name__, "-", w[0].message)

print()
print("Bornes [min, max] malgré le trou :", calc_nc.imputation_window_start_.date(), "->", calc_nc.imputation_window_end_.date())
print("Dates réellement actives (2 blocs disjoints) :", calc_nc.imputation_window_mask_.index[calc_nc.imputation_window_mask_].tolist())


Bornes [min, max] malgré le trou : 2020-03-01 -> 2020-10-01
Dates réellement actives (2 blocs disjoints) : [Timestamp('2020-03-01 00:00:00'), Timestamp('2020-04-01 00:00:00'), Timestamp('2020-09-01 00:00:00'), Timestamp('2020-10-01 00:00:00')]


### 4.8 - Avertissement : fenêtre à une seule observation (`df_single_strict`)

Après extension (ici `'strict'`, donc pas d'extension), si la fenêtre ne
contient plus qu'une seule date, un `UserWarning` invite à assouplir les
contraintes.

In [13]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    calc_single = ImputationWindowCalculator(coverage_threshold=0.9, imputation_scope='strict')
    calc_single.fit(df_single_strict)
    print("Warning :", w[0].category.__name__, "-", w[0].message)

print()
print("Fenêtre stricte :", calc_single.imputation_window_mask_.index[calc_single.imputation_window_mask_].tolist())


Fenêtre stricte : [Timestamp('2020-05-01 00:00:00')]


### 4.9 - Panel : une entite sans fenetre stricte n'empeche pas les autres (`panel_ok_bad`)

Chaque entite est traitee independamment : `BAD` (couverture disjointe,
comme 4.6) n'a pas de fenetre stricte, mais `fit` reussit tout de meme
puisque `OK` en a une. Les BORNES de `BAD` valent `None` dans les dicts par
entite ; ses lignes figurent a `False` dans la Series unique des masques
(`imputation_window_mask_`, MultiIndex `(entity..., date)`). `fit` ne leve
une `ValueError` que si **aucune** entite n'a de fenetre valide (cf. 4.12).

In [14]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    calc_ok_bad = ImputationWindowCalculator(coverage_threshold=0.5)
    calc_ok_bad.fit(panel_ok_bad)
    print("Warning :", w[0].category.__name__, "-", w[0].message)

print()
print("OK  :", calc_ok_bad.imputation_window_start_[('OK',)], "->", calc_ok_bad.imputation_window_end_[('OK',)])
print("BAD :", calc_ok_bad.imputation_window_start_[('BAD',)], "->", calc_ok_bad.imputation_window_end_[('BAD',)])
print("BAD mask entierement False :", not entity_slice(calc_ok_bad.imputation_window_mask_, ('BAD',)).any())

 UserWarning - There is no period in the DataFrame where all the variables are available. Only training_scope='unrestricted' allows training in this case.

OK  : 2020-01-01 00:00:00 -> 2020-12-01 00:00:00
BAD : None -> None
BAD mask entierement False : True


### 4.10 - Panel : entite a frequence indetectable (`panel_irregular_entity`)

`IRREGULAR` a des dates trop irregulieres pour qu'une frequence lui soit
attribuee (mais au moins 2 observations, donc pas d'erreur de detection -
voir 4.13 pour le cas `< 2` observations). `index_freq_[entity]` vaut alors
`None`, ses BORNES par entite valent `None`, et l'entite est recensee dans
`entities_without_window_`. Ses lignes figurent a `False` dans la Series des
masques ; elle est simplement absente de `coverage_by_date_`. L'entite `OK`
n'est pas affectee.

In [15]:
calc_irregular = ImputationWindowCalculator(coverage_threshold=0.5)
calc_irregular.fit(panel_irregular_entity)

print("Entites :", list(calc_irregular.imputation_window_start_))
print("entities_without_window_ :", calc_irregular.entities_without_window_)
print()
print("IRREGULAR index_freq_ :", calc_irregular.index_freq_[('IRREGULAR',)])
print("IRREGULAR bornes      :",
      calc_irregular.imputation_window_start_[('IRREGULAR',)],
      calc_irregular.imputation_window_end_[('IRREGULAR',)])
irr_mask = entity_slice(calc_irregular.imputation_window_mask_, ('IRREGULAR',))
print("IRREGULAR mask :", irr_mask.sum(), "/", len(irr_mask), "actives (toutes a False)")
print("IRREGULAR absente de coverage_by_date_ :",
      entity_slice(calc_irregular.coverage_by_date_, ('IRREGULAR',)) is None)
print()
print("OK non affectee :", calc_irregular.imputation_window_start_[('OK',)], "->", calc_irregular.imputation_window_end_[('OK',)])

Entites : [('OK',), ('IRREGULAR',)]
entities_without_window_ : (('IRREGULAR',),)

IRREGULAR index_freq_ : None
IRREGULAR bornes      : None None
IRREGULAR mask : 0 / 5 actives (toutes a False)
IRREGULAR absente de coverage_by_date_ : True

OK non affectee : 2020-01-01 00:00:00 -> 2020-12-01 00:00:00


### 4.11 - Panel : entite entierement `NaN` (`panel_empty_entity`)

Cas different de 4.10 : `EMPTY` a une frequence d'index detectable (les
**dates** sont regulieres), mais aucune colonne n'a de valeur non-`NaN`. La
grille interne (`_build_index_freq_grid`) ne peut alors se construire et vaut
`None` : `column_coverage_[entity]` vaut `None`, l'entite rejoint
`entities_without_window_` et ses lignes figurent a `False` dans les masques,
tandis que `index_freq_[entity]` reste renseigne.

In [16]:
calc_empty = ImputationWindowCalculator(coverage_threshold=0.5)
calc_empty.fit(panel_empty_entity)

print("EMPTY index_freq_       :", calc_empty.index_freq_[('EMPTY',)], "(detectable, contrairement a 4.10)")
print("EMPTY column_coverage_  :", calc_empty.column_coverage_[('EMPTY',)])
print("EMPTY dans entities_without_window_ :", ('EMPTY',) in calc_empty.entities_without_window_)
empty_mask = entity_slice(calc_empty.imputation_window_mask_, ('EMPTY',))
print("EMPTY mask entierement False :", empty_mask is not None and not empty_mask.any())
print("OK non affectee         :", calc_empty.imputation_window_start_[('OK',)] is not None)

EMPTY index_freq_       : M (detectable, contrairement a 4.10)
EMPTY column_coverage_  : None
EMPTY dans entities_without_window_ : True
EMPTY mask entierement False : True
OK non affectee         : True


### 4.12 - Panel : `ValueError` quand **aucune** entité n'a de fenêtre valide

Contrairement à §4.9 (une seule entité en échec), si toutes les entités sont
dans un des cas §4.6/§4.10/§4.11, `fit` lève une `ValueError` explicite —
aucun résultat partiel n'est retourné.

In [17]:
dates_all_bad = pd.date_range('2020-01-01', periods=12, freq='MS')
idx_all_bad = pd.MultiIndex.from_product([['A', 'B'], dates_all_bad], names=['entity', 'date'])
df_all_bad = pd.DataFrame(
    {'a': np.arange(24, dtype=float), 'b': [np.nan] * 24}, index=idx_all_bad
)  # 'b' entièrement NaN pour A ET B -> couverture conjointe jamais totale, pour aucune entité

calc_all_bad = ImputationWindowCalculator(coverage_threshold=0.5)
try:
    calc_all_bad.fit(df_all_bad)
except ValueError as e:
    print("ValueError :", e)

ValueError : No imputation window found for any entity in the panel


### 4.13 - Point de vigilance : colonne entièrement `NaN`, TS vs panel

Comportement **asymétrique**, à la source de la détection de fréquence
(`detect_dataset_frequency`, `tsforecast/utils/frequency/detector.py`), pas
d'`ImputationWindowCalculator` lui-même :
- Chemin **panel** (`_detect_panel_frequencies`) : une colonne sans
  observation valide pour une entité est explicitement tolérée, sa fréquence
  vaut `None` dans le dictionnaire retourné (documenté dans sa docstring) —
  c'est ce qui permet le cas `EMPTY` de §4.11 de passer sans erreur.
- Chemin **TS** (`_detect_time_series_frequencies`) : **aucune** tolérance
  équivalente, l'exception de `detect_time_series_frequency` («&nbsp;Series has
  only 0 non-null observations&nbsp;») remonte telle quelle. Une colonne
  entièrement `NaN` dans un DataFrame TS fait donc échouer `fit()` en entier,
  **avant même** que `min_columns` ou la logique de fenêtre n'entrent en jeu
  — alors que la même situation, transposée à une entité de panel, est gérée
  proprement (§4.11).

In [18]:
calc_min3 = ImputationWindowCalculator(min_columns=3)
df_two_usable = df_timeseries[['production_industrielle', 'inflation_ipc']].copy()
df_two_usable['colonne_vide'] = np.nan  # 3e colonne, mais entièrement vide

try:
    calc_min3.fit(df_two_usable)
except ValueError as e:
    print("ValueError (colonne TS entièrement NaN, AVANT toute logique de fenêtre) :", e)

print()
print("Rappel §4.11 : la même situation par entité, en panel, ne lève PAS d'erreur")
print("  (EMPTY index_freq_ =", calc_empty.index_freq_[('EMPTY',)], ", column_coverage_ =", calc_empty.column_coverage_[('EMPTY',)], ")")

ValueError (colonne TS entièrement NaN, AVANT toute logique de fenêtre) : Series has only 0 non-null observations, minimum required is 2

Rappel §4.11 : la même situation par entité, en panel, ne lève PAS d'erreur
  (EMPTY index_freq_ = M , column_coverage_ = None )


## 5 - `get_imputation_window_mask(data=None)`

Sans argument, renvoie directement le masque brut de `imputation_window_mask_`
(sur la grille interne, potentiellement plus large que l'index des donnees -
cf. avertissement de la docstring de classe en 0). Sur **panel**, c'est
desormais une `pd.Series` a MultiIndex `(entity..., date)` ([SPEC] 7.2), plus
un `Dict[entity, Series]`. Avec un `data`, realigne proprement le masque sur
`data.index` : c'est la facon **sure** de comparer le masque aux donnees
d'origine (comportement inchange pour les appelants passant `data`).

In [19]:
# Sans fit préalable : ValueError
try:
    ImputationWindowCalculator().get_imputation_window_mask()
except ValueError as e:
    print("ValueError (non fitted) :", e)

ValueError (non fitted) : Calculator not fitted. Call fit() first.


### 5.1 - Sans argument : identité avec `imputation_window_mask_`

In [20]:
raw_mask = calc_ts.get_imputation_window_mask()
print("Identique à l'attribut :", raw_mask.equals(calc_ts.imputation_window_mask_))

Identique à l'attribut : True


### 5.2 - Séries temporelles : alignement sur un index arbitraire

Une date hors de la grille interne (mais présente dans `data`) est
nécessairement `False` ; les dates de la fenêtre stricte, elles, restent `True`.

In [21]:
dates_align = pd.date_range('2020-01-01', periods=24, freq='MS')
df_align = pd.DataFrame({'a': np.arange(24, dtype=float), 'b': np.arange(24, dtype=float) * 2}, index=dates_align)
calc_align = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='strict')
calc_align.fit(df_align)

# Données à aligner : l'index d'origine + une date hors de la grille interne
extra_dates = dates_align.append(pd.DatetimeIndex(['2022-06-01']))
data_to_align = pd.DataFrame(index=extra_dates)

aligned = calc_align.get_imputation_window_mask(data_to_align)

print("Index identique à celui de data_to_align :", aligned.index.equals(data_to_align.index))
print("Date hors grille -> False :", aligned.loc[pd.Timestamp('2022-06-01')])
print("Fenêtre stricte -> toutes True :", aligned.loc[dates_align].all())

Index identique à celui de data_to_align : True
Date hors grille -> False : False
Fenêtre stricte -> toutes True : True


### 5.3 - Panel : alignement sur un `MultiIndex` arbitraire

Une entité absente du `fit` (`'C'` ci-dessous) reçoit `False` partout, sans
erreur — contrairement à `get_columns_with_coverage`/`get_mask_at_frequency`
(§6/§7), qui lèvent `KeyError` sur une entité inconnue *fournie
explicitement*. Ici, aucune entité n'est demandée explicitement : `data`
fournit ses propres lignes, filtrées en silence si l'entité n'a pas de masque
fitté.

In [22]:
dates_panel_align = pd.date_range('2020-01-01', periods=12, freq='MS')
idx_align = pd.MultiIndex.from_product([['A', 'B'], dates_panel_align], names=['entity', 'date'])
df_panel_align = pd.DataFrame({'a': 1.0, 'b': 1.0}, index=idx_align)
df_panel_align.loc[('A', dates_panel_align[0]), 'b'] = np.nan  # trou pour A au premier mois

calc_panel_align = ImputationWindowCalculator(coverage_threshold=0.5)
calc_panel_align.fit(df_panel_align)

extra_idx = idx_align.append(pd.MultiIndex.from_tuples([('C', dates_panel_align[0])], names=['entity', 'date']))
data_to_align_panel = pd.DataFrame(index=extra_idx)
aligned_panel = calc_panel_align.get_imputation_window_mask(data_to_align_panel)

print("Index identique :", aligned_panel.index.equals(data_to_align_panel.index))
print("Entité C (absente du fit) -> False :", aligned_panel.loc[('C', dates_panel_align[0])])
print("A, 1er mois (trou)      -> False :", aligned_panel.loc[('A', dates_panel_align[0])])
print("A, 2e mois               -> True  :", aligned_panel.loc[('A', dates_panel_align[1])])

Index identique :

 True
Entité C (absente du fit) -> False : False
A, 1er mois (trou)      -> False : False
A, 2e mois               -> True  : True


## 6 - `get_mask_at_frequency(frequency)`

Rééchantillonne le masque de la fenêtre stricte (`imputation_window_mask_`,
**pas** la version étendue par `imputation_scope`) vers une fréquence plus
basse : une période cible est `True` **ssi toutes** ses sous-périodes le
sont. Délègue à `FrequencyConverter.aggregate_to_lower_frequency(method='all')`,
avec l'offset cible ancré sur la position (S/E) de la grille source.

In [23]:
try:
    ImputationWindowCalculator().get_mask_at_frequency('QS')
except ValueError as e:
    print("ValueError (non fitted) :", e)

ValueError (non fitted) : Calculator not fitted. Call fit() first.


### 6.1 - Séries temporelles : année complète -> `True`, année partielle -> `False`

Sur `calc_align` (§5.2, fenêtre stricte mensuelle sur les 24 mois de
2020-2021, aucun trou) : les deux années sont intégralement couvertes.

In [24]:
mask_year = calc_align.get_mask_at_frequency('YS')
print(mask_year)

2020-01-01     True
2021-01-01     True
2022-01-01    False
Freq: YS-JAN, dtype: bool


In [25]:
# Un seul mois sorti de la fenêtre stricte suffit à faire basculer TOUTE l'année à False,
# même si les 11 autres mois restent dans la fenêtre
calc_partial = ImputationWindowCalculator(coverage_threshold=0.5, imputation_scope='strict')
calc_partial.fit(df_align)
calc_partial.imputation_window_mask_.loc[pd.Timestamp('2021-06-01')] = False

mask_year_partial = calc_partial.get_mask_at_frequency('YS')
print("2020 (intacte)         :", mask_year_partial.loc[pd.Timestamp('2020-01-01')])
print("2021 (1 mois retiré)   :", mask_year_partial.loc[pd.Timestamp('2021-01-01')])

2020 (intacte)         : True
2021 (1 mois retiré)   : False


### 6.2 - Ancrage de l'index résultat : toujours celui de la grille source, pas celui demandé

Demander `'YE'` (ancré en fin de période) sur une grille source `'MS'`
(ancrée en début de période) renvoie tout de même un index ancré en **début**
de période (`'YS-JAN'`) : c'est ce qui garantit un `reindex` sans perte sur
des données annuelles ancrées comme la grille source.

In [26]:
mask_ye_request = calc_align.get_mask_at_frequency('YE')
print("Fréquence demandée : 'YE' -> index résultat ancré :", mask_ye_request.index.freqstr)

# Reindex sans perte sur des données annuelles ancrées comme la grille source (MS -> YS)
yearly_data = pd.Series(
    np.arange(len(mask_ye_request)),
    index=pd.date_range(mask_ye_request.index[0], periods=len(mask_ye_request), freq='YS'),
)
print("Aucune perte au reindex :", not yearly_data.reindex(mask_ye_request.index).isna().any())

Fréquence demandée : 'YE' -> index résultat ancré : YS-JAN
Aucune perte au reindex : True


### 6.3 - Erreur : fréquence cible non strictement plus basse que la source

`get_mask_at_frequency` ne fait qu'agréger vers le **bas** : demander une
fréquence plus fine (ou égale) que celle de la grille source lève une
`ValueError`.

In [27]:
try:
    calc_align.get_mask_at_frequency('D')  # plus fin que 'M'
except ValueError as e:
    print("ValueError (cible plus fine) :", e)

ValueError (cible plus fine) : Index mask frequency : M should be higher than frequency : D


### 6.4 - Panel : fréquence commune (`str`) ou par entité (`dict`)

Comme pour `TargetFrequencyValidator`/`FrequencyAligner`, une clé scalaire
fournie par l'appelant (`'France'`) est normalisée en tuple
(`('France',)`) avant l'accès — contrairement à `TargetFrequencyValidator`,
qui exige des clés déjà en tuple (cf. `target_frequency_validator.ipynb` §3.7).

In [28]:
# Frequence commune (str) : toutes les entites converties vers la meme cible.
# get_mask_at_frequency renvoie desormais une Series a MultiIndex (entity..., date).
result_common = calc_panel.get_mask_at_frequency('YS')
for entity in calc_panel.imputation_window_start_:
    mask = entity_slice(result_common, entity)
    print(f"{entity!r:16} -> {mask.sum()}/{len(mask)} annees actives" if mask is not None else f"{entity!r:16} -> absente")

print()
# Dictionnaire par entite, cles SCALAIRES (normalisees en tuple avant acces)
result_dict = calc_panel.get_mask_at_frequency({'France': 'YS', 'Allemagne': 'QS', 'Italie': 'YS'})
for entity in calc_panel.imputation_window_start_:
    mask = entity_slice(result_dict, entity)
    print(f"{entity!r:16} -> {mask.sum()}/{len(mask)} periodes actives" if mask is not None else f"{entity!r:16} -> absente")

('Allemagne',)   -> 5/9 annees actives
('France',)      -> 5/10 annees actives
('Italie',)      -> 4/9 annees actives



('Allemagne',)   -> 20/34 periodes actives


('France',)      -> 5/10 periodes actives
('Italie',)      -> 4/9 periodes actives


### 6.5 - Panel : entite sans masque fitte -> absente du resultat, pas d'erreur

Reprend `calc_irregular` (4.10, `IRREGULAR` sans frequence detectable) :
`_convert_mask_to_frequency` retourne `None` des que le masque **ou** la
frequence source valent `None`. L'entite est alors simplement **absente** de
la Series a MultiIndex renvoyee (plus de `None` dans un dict), sans exception.

In [29]:
result_irregular = calc_irregular.get_mask_at_frequency('YS')
ok = entity_slice(result_irregular, ('OK',))
print("OK        :", ok.sum(), "/", len(ok))
print("IRREGULAR presente dans le resultat :",
      entity_slice(result_irregular, ('IRREGULAR',)) is not None)

OK        : 1 / 2
IRREGULAR presente dans le resultat : False


## 7 - `get_columns_with_coverage(start, end, entity=None)`

Filtre `column_coverage_` (bornes `[début, fin]` par colonne, calculées au
`fit`) pour ne garder que les colonnes dont la couverture chevauche
`[start, end]`.

In [30]:
try:
    ImputationWindowCalculator().get_columns_with_coverage(pd.Timestamp('2020-01-01'), pd.Timestamp('2020-06-01'))
except ValueError as e:
    print("ValueError (non fitted) :", e)

ValueError (non fitted) : Calculator not fitted. Call fit() first.


### 7.1 - Séries temporelles : plage couverte par toutes les colonnes vs plage hors couverture

Sur `calc_ts` (`df_timeseries`) : la plage `[imputation_window_start_,
imputation_window_end_]` (fenêtre stricte, par définition couverte par
TOUTES les colonnes) renvoie la liste complète ; une plage antérieure à
2015 (avant même `balance_commerciale_annuelle`) n'en renvoie aucune.

In [31]:
cols_in_window = calc_ts.get_columns_with_coverage(calc_ts.imputation_window_start_, calc_ts.imputation_window_end_)
print("Colonnes couvertes sur la fenêtre stricte :", cols_in_window)

cols_before_data = calc_ts.get_columns_with_coverage(pd.Timestamp('2010-01-01'), pd.Timestamp('2014-06-01'))
print("Colonnes couvertes avant toute donnée      :", cols_before_data)

# Plage couvrant seulement balance_commerciale_annuelle (2015-2017, avant la grille mensuelle)
cols_only_annual = calc_ts.get_columns_with_coverage(pd.Timestamp('2015-01-01'), pd.Timestamp('2017-01-01'))
print("Colonnes couvertes sur 2015-2017 (annuelle seule) :", cols_only_annual)

Colonnes couvertes sur la fenêtre stricte :

 ['production_industrielle', 'inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle']
Colonnes couvertes avant toute donnée      : []
Colonnes couvertes sur 2015-2017 (annuelle seule) : ['balance_commerciale_annuelle']


### 7.2 - Panel : `entity` fourni (liste) vs omis (dict par entité)

In [32]:
start_de, end_de = calc_panel.imputation_window_start_[('Allemagne',)], calc_panel.imputation_window_end_[('Allemagne',)]

# entity fourni : List[str]
cols_de = calc_panel.get_columns_with_coverage(start_de, end_de, entity='Allemagne')
print("Type (entity fourni)   :", type(cols_de).__name__, "->", cols_de)

# entity omis : Dict[tuple, List[str]], une clé par entité (toujours un tuple)
cols_all = calc_panel.get_columns_with_coverage(start_de, end_de)
print("Type (entity omis)      :", type(cols_all).__name__, "-> clés :", list(cols_all))

Type (entity fourni)   : list -> ['production_industrielle', 'inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'depenses_publiques_pib', 'balance_commerciale_annuelle']
Type (entity omis)      : dict -> clés : [('Allemagne',), ('France',), ('Italie',)]


### 7.3 - Panel : entité inconnue -> `KeyError` (pas de repli silencieux)

In [33]:
try:
    calc_panel.get_columns_with_coverage(pd.Timestamp('2020-01-01'), pd.Timestamp('2020-06-01'), entity='Espagne')
except KeyError as e:
    print("KeyError :", e)

KeyError : ('Espagne',)


### 7.4 - Panel : entité avec `column_coverage_` à `None` (§4.11) -> liste vide, pas d'erreur

`EMPTY` (`panel_empty_entity`, §4.11) a `column_coverage_` à `None` : la
méthode le détecte et renvoie une liste vide plutôt que de lever une
exception.

In [34]:
cols_empty = calc_empty.get_columns_with_coverage(dates_panel_edge[0], dates_panel_edge[-1], entity='EMPTY')
print("Colonnes couvertes pour EMPTY :", cols_empty)

Colonnes couvertes pour EMPTY : []


## 8 - Synthèse

| Méthode / attribut | TS | Panel |
|---|---|---|
| `imputation_window_start_`/`_end_` | `Timestamp` (ou `None`, §4.6/§4.7) | `Dict[tuple, Optional[Timestamp]]` |
| `imputation_window_mask_` (+ `imputation_strict_window_mask_`, `training_window_mask_`) | `pd.Series` sur `DatetimeIndex` | `pd.Series` unique sur MultiIndex `(entity..., date)` ([SPEC] §7.2) ; entité sans fenêtre = lignes à `False`, recensée dans `entities_without_window_` |
| `coverage_by_date_` | `pd.Series` sur `DatetimeIndex` | `pd.Series` unique sur MultiIndex `(entity..., date)`, restreinte aux entités ayant une grille |
| `index_freq_` | `str` (ou `None`) | `Dict[tuple, Optional[str]]` |
| `column_coverage_` | `Dict[str, Tuple]` (ou `None` si grille vide) | `Dict[tuple, Optional[Dict[str, Tuple]]]`, une entrée **par entité** |
| `get_imputation_window_mask(data)` | réindexée sur `data.index`, `False` hors grille | `False` pour les lignes d'entité sans masque fitté, **jamais** de `KeyError` |
| `get_mask_at_frequency` | `pd.Series`, erreur si cible pas strictement plus basse | `pd.Series` sur MultiIndex `(entity..., date)`, clés scalaires normalisées ; entité sans masque = absente |
| `get_columns_with_coverage` | `List[str]` | `List[str]` (`entity` fourni) / `Dict[tuple, List[str]]` (omis), `KeyError` si `entity` inconnu |

**Points de vigilance à retenir pour les futurs tests unitaires :**
1. **§4.13 — asymétrie TS/panel sur une colonne entièrement `NaN`** : côté TS,
   `fit()` lève une `ValueError` (`detect_time_series_frequency` ne tolère
   aucune colonne à 0 observation valide) ; côté panel, la même situation par
   entité est tolérée nativement par `_detect_panel_frequencies`
   (fréquence `None`, géré ensuite par la branche `column_coverage_ = None`
   de `_compute_window`). Un test de non-régression devrait épingler les
   DEUX comportements, pas seulement le panel (déjà couvert par
   `test_column_coverage_is_per_entity_for_panel` et voisins).
2. **§4.10 vs §4.11 — deux causes distinctes de `None` en panel, mêmes
   symptômes en apparence** : `index_freq_[entity] is None` (fréquence de
   l'INDEX indétectable, dates trop irrégulières, §4.10) court-circuite tout
   calcul de fenêtre dans `_fit_panel` ; `column_coverage_[entity] is None`
   (grille interne vide car AUCUNE valeur non-`NaN` sur aucune colonne,
   §4.11) est un cas différent, où `index_freq_[entity]` reste renseigné. Les
   deux se distinguent seulement en inspectant `index_freq_` : un test qui ne
   vérifierait que `imputation_window_mask_ is None` les confondrait.
3. **§4.4/§4.5 — extension contiguë** : `extended_backward`/`extended_forward`
   s'arrêtent au premier point sous le seuil, n'activent jamais une date
   au-delà d'un trou même si elle satisferait individuellement
   `coverage_threshold`. Sensible à l'ORDRE des colonnes du DataFrame (la
   colonne utilisée pour détecter la position S/E de la grille doit être
   listée en premier si elle a des trous — bug latent hors périmètre de
   `_build_index_freq_grid`, déjà noté dans
   `tests/frequency/test_imputation_window.py::TestExtensionContiguity`).
4. **§5 — ne jamais intersecter `imputation_window_mask_` directement** avec
   un masque calculé sur les données de l'appelant : la grille interne peut
   déborder de la dernière ligne réelle (observation basse fréquence
   couvrant plusieurs sous-périodes). Toujours passer par
   `get_imputation_window_mask(data)`.
5. **§6.2 — ancrage de `get_mask_at_frequency`** : l'index résultat reste
   ancré comme la grille SOURCE, jamais comme la fréquence cible demandée
   (`'YE'` demandé -> `'YS-JAN'` obtenu ici) — un test qui vérifierait
   `result.index.freqstr == target_frequency` échouerait à tort.
6. **§6.1 — `get_mask_at_frequency` porte sur la fenêtre STRICTE**
   (`imputation_window_mask_` avant extension), quel que soit
   `imputation_scope` fourni au constructeur : c'est la même série que celle
   utilisée pour dériver `imputation_window_start_`/`_end_`, pas la version
   étendue visible via `get_imputation_window_mask()` après un scope
   `extended_*`.
7. **§5.3 vs §7.3 — deux politiques différentes pour une entité inconnue** :
   `get_imputation_window_mask(data)` ignore silencieusement (renvoie
   `False`) une entité présente dans `data` mais absente du `fit`, alors que
   `get_columns_with_coverage`/`get_mask_at_frequency` lèvent `KeyError` pour
   une entité explicitement demandée par l'appelant et absente — cohérent
   avec la distinction documentée dans la docstring de classe (§3.4/§5.4 de
   la revue HFI), mais à couvrir explicitement par un test pour chaque
   méthode.